# Phase 7: Loss Given Default (LGD) Modeling
This notebook implements Loss Given Default (LGD) modeling. LGD estimates the severity of loss when a loan defaults. Target is defined as: `LGD = (loan_amnt - recoveries) / loan_amnt` capped to `[0.0, 1.0]`.


In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from lgd_model import LGDModel


## 1. Load Data Splits
Load the preprocessed default datasets for model training.


In [2]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


Train size: (18061, 116)
OOT size: (20516, 116)


## 2. Train LGD Models
Instantiate and fit LGD models (XGBoost Regressor and Random Forest benchmark) on historical default occurrences.


In [3]:
lgd_model = LGDModel()
metrics = lgd_model.fit(train_df, oot_df)
print('Model Evaluation Metrics:')
print(metrics)


Preparing LGD Training Data...
LGD Training cohort size: 2371 defaults
Preparing LGD Validation Data...
LGD Validation cohort size: 20516 defaults
Training LGD XGBoost Regressor...
Training LGD Random Forest Regressor (Benchmark)...

LGD Model Evaluation Summary:
XGBoost Validation - RMSE: 0.0728, MAE: 0.0542, R2: -1.5835
Random Forest Validation - RMSE: 0.0667, MAE: 0.0516, R2: -1.1662
Model Evaluation Metrics:
{'xgb': {'train_rmse': np.float64(0.09921878934184437), 'train_mae': 0.049971771072016155, 'train_r2': 0.42815947290514567, 'val_rmse': np.float64(0.07281552083044408), 'val_mae': 0.054189761536022014, 'val_r2': -1.5834964000839733}, 'rf': {'train_rmse': np.float64(0.11604458161881673), 'train_mae': 0.056253491342253485, 'train_r2': 0.21776576668965508, 'val_rmse': np.float64(0.06667571128786101), 'val_mae': 0.0515813123040957, 'val_r2': -1.1661835575013826}}


## 3. Generate Predictions & Reports
Calculate predictions for the OOT validation set, evaluate error residuals, save the pickled model file, and compile the final PDF model documentation report.


In [4]:
# Predict on OOT
oot_defaults = oot_df[oot_df['loan_status'].isin(['Charged Off', 'Default'])].copy()
oot_defaults['pred_lgd'] = lgd_model.predict_lgd(oot_defaults)
oot_defaults['actual_lgd'] = lgd_model.calculate_lgd_target(oot_defaults)
print(oot_defaults[['actual_lgd', 'pred_lgd']].describe())

# Save outputs
lgd_model.save_model('outputs/scorecards/lgd_model.pkl')
oot_defaults[['id', 'member_id', 'actual_lgd', 'pred_lgd']].to_csv('outputs/scorecards/lgd_predictions.csv', index=False)
lgd_model.generate_report(train_df, oot_df, 'outputs/reports/lgd_model_report.pdf', metrics)
print('LGD predictions and PDF report generated successfully.')


        actual_lgd     pred_lgd
count  3256.000000  3256.000000
mean      0.946245     0.946047
std       0.102488     0.038460
min       0.000000     0.594442
25%       0.947854     0.939148
50%       0.964927     0.955325
75%       0.999102     0.966730
max       1.000000     1.000000
LGD Model successfully saved to outputs/scorecards/lgd_model.pkl
LGD Model report successfully generated at outputs/reports/lgd_model_report.pdf
LGD predictions and PDF report generated successfully.
